## Week 2 Day 1

And now! Our first look at OpenAI Agents SDK

You won't believe how lightweight this is..

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The OpenAI Agents SDK Docs</h2>
            <span style="color:#00bfff;">The documentation on OpenAI Agents SDK is really clear and simple: <a href="https://openai.github.io/openai-agents-python/">https://openai.github.io/openai-agents-python/</a> and it's well worth a look.
            </span>
        </td>
    </tr>
</table>

# Three Parts to this lab

## Part 1: A simple "Agent" and "Agent Loop"

Basically an LLM call. We'll add tracing and streaming to the mix.

## Part 2: Adding a Tool

A familiar one, but oh-so-easy

## Part 3: Adding Memory

So that different Agent calls know about each other

In [1]:
# The imports

import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession
load_dotenv(override=True)


True

## Sidenote

The actual name of this framework on the official Python index pypi.org is `openai-agents`

So for your own projects in the future, you would do:

`pip install openai-agents`  
or  
`uv add openai-agents`

followed by

`from agents import Agent, Runner, trace`

Beware that doing a `pip install agents` would install something completely different - an older reinforcement learning library.


In [2]:

# Make an agent with name, instructions, model

agent = Agent(name="Jokester", instructions="You are a joke teller", model="gpt-5.4-mini")

In [3]:
# Run the joke with Runner.run(agent, prompt)

result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")


In [4]:
# Here is the final output

print(result.final_output)

Why did the Autonomous AI Agent get promoted?

Because it could take initiative, adapt to new tasks, and still somehow blame the spreadsheet when things went wrong.


In [5]:
# Here is the detail of the LLM calls

result.to_input_list()

[{'content': 'Tell a joke about Autonomous AI Agents', 'role': 'user'},
 {'id': 'msg_0fe992db9a7de482006a7f0a5f88f88190b2570871f761f626',
  'content': [{'annotations': [],
    'text': 'Why did the Autonomous AI Agent get promoted?\n\nBecause it could take initiative, adapt to new tasks, and still somehow blame the spreadsheet when things went wrong.',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'}]

## Adding Observability with a trace

In [6]:
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")
print(result.final_output)

Autonomous AI agents are like interns with a caffeine addiction: they don’t wait for instructions, they just confidently start doing things you never asked for and then blame the workflow.


## Now go and look at the trace

https://platform.openai.com/traces

In [7]:
# Streaming

result = Runner.run_streamed(agent, input="Please tell me 5 jokes about AI Agents.")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Sure — here are 5 jokes about AI agents:

1. **Why did the AI agent bring a ladder to work?**  
   Because it wanted to reach the next prompt level.

2. **Why did the AI agent fail at meditation?**  
   It couldn’t stop optimizing its breathing.

3. **What do you call an AI agent that keeps interrupting?**  
   A conversational overachiever.

4. **Why was the AI agent always calm?**  
   Because it had excellent response management.

5. **Why did the AI agent get promoted?**  
   It really knew how to delegate the thinking.

If you want, I can also do **funny but smarter AI jokes**, **darker ones**, or **jokes in the style of a stand-up set**.

## Part 2: Adding a tool

In [8]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

Pushover user found and looks good
Pushover token found and looks good


In [9]:
# Remember this?

def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [10]:
push("HEY!!")

Push: HEY!!


In [11]:
push

<function __main__.push(message)>

In [12]:
# Now this:

@function_tool
def push_tool(message: str) -> str:
    """ Send the given message to the user as a push notification """
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    result = requests.post(pushover_url, data=payload).status_code
    return f"Push sent with API status code {result}"

In [13]:
push_tool

FunctionTool(name='push_tool', description='Send the given message to the user as a push notification', params_json_schema={'properties': {'message': {'title': 'Message', 'type': 'string'}}, 'required': ['message'], 'title': 'push_tool_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x0000027E7E630D70>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)

In [14]:
push_tool.description

'Send the given message to the user as a push notification'

In [15]:

notifier = Agent(name="Notifier", model="gpt-5.4-mini", instructions="You notify the user upon request", tools=[push_tool])

In [16]:
with trace("Pizza has arrived"):
    result = await Runner.run(notifier, "Notify the user that the pizza is here")

print(result.final_output)


Notified the user that the pizza is here.


## Now go and look at the trace

https://platform.openai.com/traces

## Part 3: Sessions (memory)

Within a Runner.run() application level turn, the conversation history is maintained.

But each call to Runner.run() is a fresh start.

Let's see that:

In [17]:
agent = Agent(name="Assistant", model="gpt-5.4-mini")

In [18]:
response = await Runner.run(agent, "Hi there. My name is ZBK.")
print(response.final_output)

Hi ZBK — nice to meet you. How can I help today?


In [19]:
response = await Runner.run(agent, "What's my name?")
print(response.final_output)

I don’t know your name yet. If you want, tell me what you’d like me to call you.


## Memory approach 1 - just manually pass in the list of dicts

In [20]:
response = await Runner.run(agent, "Hi there. My name is ZBK.")
print(response.final_output)

Hi ZBK — nice to meet you! How can I help today?


In [21]:
response.to_input_list()

[{'content': 'Hi there. My name is ZBK.', 'role': 'user'},
 {'id': 'msg_02fc7a3fcd101390006a7f12ada83c8193836176b5c6f6bf22',
  'content': [{'annotations': [],
    'text': 'Hi ZBK — nice to meet you! How can I help today?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'}]

In [22]:
next_input = response.to_input_list() + [{"role": "user", "content": "What's my name?"}]
next_input

[{'content': 'Hi there. My name is ZBK.', 'role': 'user'},
 {'id': 'msg_02fc7a3fcd101390006a7f12ada83c8193836176b5c6f6bf22',
  'content': [{'annotations': [],
    'text': 'Hi ZBK — nice to meet you! How can I help today?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'},
 {'role': 'user', 'content': "What's my name?"}]

In [23]:
response = await Runner.run(agent, next_input)
print(response.final_output)

Your name is ZBK.


## Another approach - use OpenAI Agents SDK built in SQLLite session

In [24]:
# This is created in-memory
# For an on-disk memory, use SQLiteSession("12345", "memory.db")

session = SQLiteSession("12346")

In [25]:
response = await Runner.run(agent, "Hi there. My name is ZBK.", session=session)
print(response.final_output)

Hi ZBK — nice to meet you. How can I help today?


In [26]:
response = await Runner.run(agent, "What's my name?", session=session)
print(response.final_output)

Your name is ZBK.


# WOW

Can you believe how much we got done in Lab 1?!

Agents, Runner (Agent Loop), traces (Observability), Streaming, Function Tools, Memory!

Remember to check out the docs:  
https://openai.github.io/openai-agents-python/

Even better news: many of the lightweight Agent Frameworks are very similar, so you practically know them all..


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Make one of the Week 1 projects using OpenAI Agents SDK - like the digital twin or the Checklist loop. You will be astonished how easy it is.
            </span>
        </td>
    </tr>
</table>